# ML Portfolio Optimization - Baseline Experiments (No Sentiment)

**Goal**: Compare ML methods (Ridge vs LightGBM) against baseline strategies WITHOUT sentiment features.

**Strategies to test:**
1. Equal Weight (baseline)
2. Mean-Variance (baseline)
3. 60/40 Static (baseline)
4. Predictive Sharpe + Ridge (ML)
5. Predictive Sharpe + LightGBM (ML)

**ETFs**: SPY, QQQ, VTI, TLT, BND, GLD, VEA, VWO, IWM, XLE (10 total)

**Metrics**: Sharpe Ratio, Annualized Return, Volatility, Max Drawdown, Sortino, Calmar

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from data import load_default_etfs
from strategies import (
    EqualWeightStrategy,
    MeanVarianceStrategy,
    StaticStrategy,
    PredictiveSharpeStrategy,
    GradientBoostingSharpeStrategy
)
from backtest import Backtester
from metrics import PerformanceMetrics

# Plotting config
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

OSError: dlopen(/Users/vincent/Programming_Projects/ETF-Optimization/.venv/lib/python3.11/site-packages/lightgbm/lib/lib_lightgbm.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib
  Referenced from: <8FC36893-94B8-343C-9D9F-4CCBFE81B89B> /Users/vincent/Programming_Projects/ETF-Optimization/.venv/lib/python3.11/site-packages/lightgbm/lib/lib_lightgbm.dylib
  Reason: tried: '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/li

## 1. Load Data (10 ETFs)

In [ ]:
# Load 10 ETFs
tickers = ['SPY', 'QQQ', 'VTI', 'TLT', 'BND', 'GLD', 'VEA', 'VWO', 'IWM', 'XLE']

print(f"Loading data for {len(tickers)} ETFs...")
data = load_default_etfs(
    tickers=tickers,
    start='2015-01-01',
    end='2025-12-07'
)

print(f"\nData loaded: {len(data)} days")
print(f"Date range: {data.index[0]} to {data.index[-1]}")
print(f"\nTickers: {tickers}")
print(f"\nSample data:")
print(data.head())

## 2. Split Data (Train/Val/Test)

In [ ]:
# Define splits (60% train, 20% val, 20% test)
n = len(data)
train_end_idx = int(n * 0.6)
val_end_idx = int(n * 0.8)

train_data = data.iloc[:train_end_idx]
val_data = data.iloc[train_end_idx:val_end_idx]
test_data = data.iloc[val_end_idx:]

print(f"Train: {train_data.index[0]} to {train_data.index[-1]} ({len(train_data)} days)")
print(f"Val:   {val_data.index[0]} to {val_data.index[-1]} ({len(val_data)} days)")
print(f"Test:  {test_data.index[0]} to {test_data.index[-1]} ({len(test_data)} days)")

## 3. Initialize Strategies

In [ ]:
# Baseline strategies
equal_weight = EqualWeightStrategy()
mean_variance = MeanVarianceStrategy(risk_free_rate=0.02)
static_6040 = StaticStrategy(weights={'SPY': 0.6, 'TLT': 0.4})  # 60/40 portfolio

# ML strategies (NO sentiment features)
ridge_ml = PredictiveSharpeStrategy(
    lookback_days=252,  # 1 year of history
    min_history=60,
    risk_free_rate=0.02
)

lightgbm_ml = GradientBoostingSharpeStrategy(
    lookback_days=252,
    min_history=60,
    risk_free_rate=0.02,
    n_estimators=100,
    max_depth=5,
    learning_rate=0.05
)

strategies = {
    'Equal Weight': equal_weight,
    'Mean-Variance': mean_variance,
    '60/40 Static': static_6040,
    'Ridge ML': ridge_ml,
    'LightGBM ML': lightgbm_ml
}

print(f"Initialized {len(strategies)} strategies")
for name in strategies:
    print(f"  - {name}")

## 4. Run Backtests

In [ ]:
# Run backtests on TEST set (out-of-sample)
results = {}

print("Running backtests on TEST set...\n")
print("="*80)

for name, strategy in strategies.items():
    print(f"\nBacktesting: {name}")
    print("-"*80)
    
    backtester = Backtester(
        strategy=strategy,
        data=test_data,
        initial_capital=100000,
        transaction_cost=0.001  # 10 bps
    )
    
    # Run backtest
    portfolio_values, weights_history = backtester.run()
    
    # Compute metrics
    metrics = PerformanceMetrics(portfolio_values)
    
    results[name] = {
        'portfolio_values': portfolio_values,
        'weights_history': weights_history,
        'metrics': metrics
    }
    
    print(f"  Sharpe Ratio: {metrics.sharpe_ratio():.4f}")
    print(f"  Annual Return: {metrics.annualized_return():.2%}")
    print(f"  Volatility: {metrics.volatility():.2%}")
    print(f"  Max Drawdown: {metrics.max_drawdown():.2%}")

print("\n" + "="*80)
print("Backtests complete!")

## 5. Results Summary Table

In [ ]:
# Create summary DataFrame
summary_data = []

for name, result in results.items():
    m = result['metrics']
    summary_data.append({
        'Strategy': name,
        'Sharpe': m.sharpe_ratio(),
        'Annual Return': m.annualized_return(),
        'Volatility': m.volatility(),
        'Max Drawdown': m.max_drawdown(),
        'Sortino': m.sortino_ratio(),
        'Calmar': m.calmar_ratio()
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Sharpe', ascending=False)

print("\n" + "="*80)
print("PERFORMANCE SUMMARY (TEST SET)")
print("="*80)
print(summary_df.to_string(index=False))

# Highlight best performer
best_strategy = summary_df.iloc[0]['Strategy']
best_sharpe = summary_df.iloc[0]['Sharpe']
print(f"\n🏆 Best Strategy: {best_strategy} (Sharpe: {best_sharpe:.4f})")

## 6. Plot Portfolio Values

In [ ]:
# Plot cumulative returns
plt.figure(figsize=(14, 8))

for name, result in results.items():
    pv = result['portfolio_values']
    plt.plot(pv.index, pv.values, label=name, linewidth=2)

plt.title('Portfolio Value Over Time (Test Set)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Portfolio Value ($)', fontsize=12)
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Plot Drawdowns

In [ ]:
# Plot drawdowns
fig, axes = plt.subplots(len(results), 1, figsize=(14, 4*len(results)))
if len(results) == 1:
    axes = [axes]

for (name, result), ax in zip(results.items(), axes):
    pv = result['portfolio_values']
    
    # Compute drawdown
    cummax = pv.cummax()
    drawdown = (pv - cummax) / cummax
    
    ax.fill_between(drawdown.index, 0, drawdown.values, alpha=0.3, color='red')
    ax.plot(drawdown.index, drawdown.values, color='darkred', linewidth=1)
    ax.set_title(f'{name} - Drawdown', fontsize=12, fontweight='bold')
    ax.set_ylabel('Drawdown', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.show()

## 8. Compare Sharpe Ratios (Bar Chart)

In [ ]:
# Bar chart of Sharpe ratios
plt.figure(figsize=(12, 6))

colors = ['#2ecc71' if 'ML' in name else '#3498db' for name in summary_df['Strategy']]
bars = plt.bar(summary_df['Strategy'], summary_df['Sharpe'], color=colors, alpha=0.8, edgecolor='black')

plt.title('Sharpe Ratio Comparison (No Sentiment)', fontsize=16, fontweight='bold')
plt.ylabel('Sharpe Ratio', fontsize=12)
plt.xlabel('Strategy', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.axhline(y=1.0, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Sharpe = 1.0')
plt.grid(True, alpha=0.3, axis='y')
plt.legend()
plt.tight_layout()
plt.show()

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

## 9. Key Insights

**Questions to answer:**
1. Do ML methods (Ridge/LightGBM) beat baseline strategies?
2. Does LightGBM beat Ridge?
3. What is the best Sharpe ratio achieved (target: >1.5)?
4. Are ML methods overfitting (check train vs test performance)?

**Next steps:**
- If ML methods underperform → debug feature engineering
- If ML methods work → add sentiment features and compare

In [ ]:
# Print key insights
print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)

# Best strategy
best = summary_df.iloc[0]
print(f"\n1. Best Strategy: {best['Strategy']}")
print(f"   - Sharpe: {best['Sharpe']:.4f}")
print(f"   - Annual Return: {best['Annual Return']:.2%}")
print(f"   - Max Drawdown: {best['Max Drawdown']:.2%}")

# ML vs Baseline
ml_strategies = summary_df[summary_df['Strategy'].str.contains('ML')]
baseline_strategies = summary_df[~summary_df['Strategy'].str.contains('ML')]

print(f"\n2. ML vs Baseline:")
print(f"   - Best ML Sharpe: {ml_strategies['Sharpe'].max():.4f}")
print(f"   - Best Baseline Sharpe: {baseline_strategies['Sharpe'].max():.4f}")
print(f"   - Improvement: {(ml_strategies['Sharpe'].max() - baseline_strategies['Sharpe'].max()):.4f}")

# Ridge vs LightGBM
ridge_sharpe = summary_df[summary_df['Strategy'] == 'Ridge ML']['Sharpe'].values[0]
lgbm_sharpe = summary_df[summary_df['Strategy'] == 'LightGBM ML']['Sharpe'].values[0]

print(f"\n3. Ridge vs LightGBM:")
print(f"   - Ridge Sharpe: {ridge_sharpe:.4f}")
print(f"   - LightGBM Sharpe: {lgbm_sharpe:.4f}")
print(f"   - Winner: {'LightGBM' if lgbm_sharpe > ridge_sharpe else 'Ridge'}")

print("\n" + "="*80)